# Clase 10 · Práctica: aprendizaje no supervisado y representaciones

En esta práctica aplicas lo que viste en la Clase 10 sobre aprendizaje no supervisado y representaciones. Trabajas con un autocodificador (*autoencoder*), la red que comprime una imagen en un vector latente y la reconstruye a partir de él, y usas el error de reconstrucción para detectar ejemplos anómalos.

## Qué vas a practicar

- Completar la arquitectura de un autocodificador con capas lineales en PyTorch y entrenarlo sobre el conjunto de dígitos manuscritos MNIST (Modified National Institute of Standards and Technology)
- Usar el error de reconstrucción de un autocodificador ya entrenado para detectar las imágenes de test peor reconstruidas, un indicio de que son anómalas o difíciles

## Requisito previo

Antes de empezar, asegúrate de haber visto la Clase 10 completa: necesitas entender qué es un codificador, un decodificador, el cuello de botella latente y por qué el error de reconstrucción sirve como señal de anomalía.

Cada ejercicio trae el andamiaje de código ya escrito. Completa solo las partes marcadas con `# TODO`. Cuando termines, despliega la solución para comparar tu código, pero intenta resolverlo por tu cuenta primero.

Este cuaderno está pensado para Google Colab y solo usa librerías que ya vienen instaladas: PyTorch, torchvision y matplotlib. Si Colab te asigna una GPU (unidad de procesamiento gráfico), el entrenamiento es más rápido, pero el cuaderno funciona igual solo con CPU (unidad central de procesamiento).

## Ejercicio 1 — Detecta dígitos anómalos con un autocodificador

### El caso

Trabajas en un equipo que digitaliza formularios manuscritos y quiere detectar automáticamente los dígitos mal escaneados o directamente ilegibles, antes de que lleguen a un sistema de reconocimiento óptico de caracteres (OCR). No tienes etiquetas de "normal" o "anómalo": solo un archivo con miles de dígitos escaneados. La solución no supervisada es entrenar un autocodificador sobre los dígitos normales; los que la red reconstruye peor son los candidatos más probables a ser anómalos o estar mal escaneados.

En este ejercicio completas la arquitectura del autocodificador y lo entrenas sobre MNIST. En el ejercicio 2 usas ese modelo entrenado para identificar los peores casos.

### Lo que tienes que hacer

Completa la clase `Autoencoder`:

- El **codificador** (`self.codificador`) es un `nn.Sequential` que comprime progresivamente el vector de entrada: de `dim_entrada` a 256, de 256 a 128 y de 128 a `dim_latente`, con una activación `nn.ReLU()` entre cada capa lineal
- El **decodificador** (`self.decodificador`) hace el camino inverso: de `dim_latente` a 128, de 128 a 256 y de 256 a `dim_entrada`, también con `nn.ReLU()` entre capas, y una `nn.Sigmoid()` final, porque los píxeles de MNIST están normalizados entre cero y uno
- El método `forward` pasa `x` por el codificador para obtener el vector latente `z`, y `z` por el decodificador para producir la reconstrucción

El resto del andamiaje —carga de datos, función de pérdida, optimizador y bucle de entrenamiento— ya está escrito. No lo modifiques, solo complétalo con tu clase.

**Pista**: fíjate en el ejemplo de código comentado que viste en el capítulo del manual. La arquitectura que te pide este ejercicio es exactamente esa.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# Usa GPU si Colab te la ha asignado, si no, usa CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de entrenamiento: {device}")

# --- Datos ---
# MNIST: dígitos de 28x28 en escala de grises, valores normalizados entre 0 y 1
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

DIM_ENTRADA = 28 * 28   # cada imagen se aplana a un vector de 784 valores
DIM_LATENTE = 32        # tamaño del cuello de botella: el "resumen" de cada imagen


# --- Definición del autocodificador ---
class Autoencoder(nn.Module):
    def __init__(self, dim_entrada=DIM_ENTRADA, dim_latente=DIM_LATENTE):
        super().__init__()
        # TODO: define el codificador como nn.Sequential.
        # Comprime dim_entrada -> 256 -> 128 -> dim_latente, con nn.ReLU()
        # entre cada capa nn.Linear.
        self.codificador = ...

        # TODO: define el decodificador como nn.Sequential.
        # Expande dim_latente -> 128 -> 256 -> dim_entrada, con nn.ReLU()
        # entre capas lineales y una nn.Sigmoid() final (píxeles en [0, 1]).
        self.decodificador = ...

    def forward(self, x):
        # TODO: calcula z pasando x por el codificador, y x_reconstruida
        # pasando z por el decodificador.
        z = ...
        x_reconstruida = ...
        return x_reconstruida, z

In [ ]:
modelo = Autoencoder().to(device)
print(modelo)

# El error de reconstrucción binario funciona bien con píxeles en [0, 1] y salida sigmoide
criterio = nn.BCELoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)


def entrenar_una_epoca(modelo, loader, criterio, optimizador):
    modelo.train()
    perdida_total = 0.0
    for imagenes, _ in loader:  # las etiquetas no se usan: no hace falta supervisión
        imagenes = imagenes.view(imagenes.size(0), -1).to(device)  # aplana a vector

        optimizador.zero_grad()
        reconstruidas, _ = modelo(imagenes)
        perdida = criterio(reconstruidas, imagenes)
        perdida.backward()
        optimizador.step()

        perdida_total += perdida.item() * imagenes.size(0)
    return perdida_total / len(loader.dataset)


num_epocas = 10
for epoca in range(1, num_epocas + 1):
    perdida_train = entrenar_una_epoca(modelo, train_loader, criterio, optimizador)
    print(f"Época {epoca}/{num_epocas} | pérdida de reconstrucción: {perdida_train:.4f}")

In [ ]:
# Comprobación visual rápida: compara diez originales con sus reconstrucciones
modelo.eval()
imagenes_prueba, _ = next(iter(test_loader))
imagenes_prueba = imagenes_prueba[:10]

with torch.no_grad():
    entrada_plana = imagenes_prueba.view(imagenes_prueba.size(0), -1).to(device)
    reconstruidas, _ = modelo(entrada_plana)
    reconstruidas = reconstruidas.view(-1, 28, 28).cpu()

fig, ejes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    ejes[0, i].imshow(imagenes_prueba[i].squeeze(), cmap="gray")
    ejes[0, i].axis("off")
    ejes[1, i].imshow(reconstruidas[i], cmap="gray")
    ejes[1, i].axis("off")

ejes[0, 0].set_title("Original", loc="left", fontsize=10)
ejes[1, 0].set_title("Reconstruida", loc="left", fontsize=10)
plt.tight_layout()
plt.show()

<details>
<summary>💡 Ver solución</summary>

```python
class Autoencoder(nn.Module):
    def __init__(self, dim_entrada=DIM_ENTRADA, dim_latente=DIM_LATENTE):
        super().__init__()
        # Codificador: comprime progresivamente hasta el cuello de botella
        self.codificador = nn.Sequential(
            nn.Linear(dim_entrada, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, dim_latente),
        )
        # Decodificador: expande de vuelta hasta el tamaño original
        # Sigmoid final porque los píxeles de MNIST están normalizados en [0, 1]
        self.decodificador = nn.Sequential(
            nn.Linear(dim_latente, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, dim_entrada),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.codificador(x)
        x_reconstruida = self.decodificador(z)
        return x_reconstruida, z
```
</details>

## Ejercicio 2 — Encuentra las diez imágenes peor reconstruidas

### El caso

Ya tienes el autocodificador entrenado del ejercicio 1. Ahora te toca la parte que de verdad interesa al equipo: convertir el error de reconstrucción en una lista concreta de sospechosos. La idea es simple: cuanto mayor es el error de reconstrucción, menos se parece el dígito a lo que la red vio durante el entrenamiento, y más probable es que sea anómalo o esté mal escaneado.

### Lo que tienes que hacer

Completa la función `calcular_errores_reconstruccion`, que recorre el conjunto de test por lotes y calcula el error cuadrático medio (MSE) de cada imagen, no del lote completo.

- Dentro del bucle, obtén la reconstrucción del lote con el modelo ya entrenado
- Calcula el error por imagen. Eleva al cuadrado la diferencia entre la entrada aplanada y la reconstrucción, y reduce con `.mean(dim=1)`: así obtienes un único valor por imagen, no un valor por lote
- Tras calcular `errores_test` para todo el conjunto de test, usa `torch.topk` para obtener los índices de las diez imágenes con mayor error

**Pista**: `nn.functional.mse_loss` con `reduction="none"` también sirve para obtener el error sin reducir, pero aquí es más directo hacerlo a mano con `** 2` y `.mean(dim=1)`.

In [ ]:
def calcular_errores_reconstruccion(modelo, dataset, device):
    """
    Calcula el error de reconstrucción (MSE) de cada imagen del dataset.

    Devuelve:
        errores: tensor 1D de forma (N,), un valor de error por imagen
        imagenes: tensor de forma (N, 28, 28) con las imágenes originales
    """
    modelo.eval()
    loader = DataLoader(dataset, batch_size=256, shuffle=False)

    errores = []
    imagenes_todas = []

    with torch.no_grad():
        for imagenes, _ in loader:
            entrada_plana = imagenes.view(imagenes.size(0), -1).to(device)

            # TODO: obtén la reconstrucción del modelo para este lote
            reconstruidas, _ = ...

            # TODO: calcula el error cuadrático medio POR IMAGEN.
            # Reduce sobre la dimensión de píxeles (dim=1), no sobre el lote:
            # el resultado debe tener forma (tamaño_del_lote,), un valor por imagen.
            error_por_imagen = ...

            errores.append(error_por_imagen.cpu())
            imagenes_todas.append(imagenes.view(-1, 28, 28))

    errores = torch.cat(errores)
    imagenes_todas = torch.cat(imagenes_todas)
    return errores, imagenes_todas


errores_test, imagenes_test = calcular_errores_reconstruccion(modelo, test_dataset, device)
print(f"Error medio de reconstrucción en test: {errores_test.mean():.4f}")

In [ ]:
N_PEORES = 10

# TODO: obtén los índices de las N_PEORES imágenes con mayor error de
# reconstrucción a partir de 'errores_test'. Pista: torch.topk(tensor, k)
# devuelve (valores, índices) ordenados de mayor a menor.
indices_peores = ...

fig, ejes = plt.subplots(1, N_PEORES, figsize=(15, 2))
for i, idx in enumerate(indices_peores):
    ejes[i].imshow(imagenes_test[idx], cmap="gray")
    ejes[i].set_title(f"{errores_test[idx]:.3f}", fontsize=8)
    ejes[i].axis("off")

plt.suptitle("Las diez imágenes con mayor error de reconstrucción (candidatas a anómalas)")
plt.tight_layout()
plt.show()

<details>
<summary>💡 Ver solución</summary>

```python
def calcular_errores_reconstruccion(modelo, dataset, device):
    modelo.eval()
    loader = DataLoader(dataset, batch_size=256, shuffle=False)

    errores = []
    imagenes_todas = []

    with torch.no_grad():
        for imagenes, _ in loader:
            entrada_plana = imagenes.view(imagenes.size(0), -1).to(device)

            reconstruidas, _ = modelo(entrada_plana)

            # Error cuadrático medio por imagen: se reduce sobre los píxeles (dim=1)
            error_por_imagen = ((entrada_plana - reconstruidas) ** 2).mean(dim=1)

            errores.append(error_por_imagen.cpu())
            imagenes_todas.append(imagenes.view(-1, 28, 28))

    errores = torch.cat(errores)
    imagenes_todas = torch.cat(imagenes_todas)
    return errores, imagenes_todas


errores_test, imagenes_test = calcular_errores_reconstruccion(modelo, test_dataset, device)
print(f"Error medio de reconstrucción en test: {errores_test.mean():.4f}")

N_PEORES = 10
valores_peores, indices_peores = torch.topk(errores_test, N_PEORES)

fig, ejes = plt.subplots(1, N_PEORES, figsize=(15, 2))
for i, idx in enumerate(indices_peores):
    ejes[i].imshow(imagenes_test[idx], cmap="gray")
    ejes[i].set_title(f"{errores_test[idx]:.3f}", fontsize=8)
    ejes[i].axis("off")

plt.suptitle("Las diez imágenes con mayor error de reconstrucción (candidatas a anómalas)")
plt.tight_layout()
plt.show()
```
</details>

## Cierre

Acabas de convertir una red que solo aprende a copiar su propia entrada en un detector de anomalías sin una sola etiqueta. Ese mismo patrón —entrenar sobre datos normales y usar el error de reconstrucción como señal— se aplica a detección de fraude, mantenimiento predictivo o control de calidad industrial, no solo a dígitos manuscritos.

Antes de pasar al siguiente capítulo, prueba a cambiar `DIM_LATENTE` y observa cómo se desplaza la lista de las diez imágenes peor reconstruidas: un cuello de botella más estrecho suele marcar como anómalos hasta dígitos razonablemente normales, porque el modelo pierde capacidad para representarlos bien.

**Guarda tu cuaderno completado y compártelo con tu equipo: ver qué dígitos señala cada modelo como "más anómalo" es la mejor forma de intuir qué aprende realmente el espacio latente.**